# Used Car Price Prediction — India, 2025–26
Walkthrough of the data and the model. The production code is in `src/` (`build_dataset.py`, `features.py`, `train.py`).

In [1]:
import sys, json, numpy as np, pandas as pd
sys.path.insert(0, 'src')
from features import add_features, CarPriceModel, FEATURES, TARGET
df = pd.read_csv('data/used_cars_latest.csv')
print(df.shape)
df.head()

(40320, 14)


,Source,Platform,Listing_Year,Brand,Model,Variant,Variant_Key,Year,Kms_Driven,Fuel_Type,Transmission,Owner,City,Selling_Price
0,cardekho_2025,CarDekho,2025,Ashok Leyland,Stile,LE 7 SEATER,7 le seater,2014,100000,Diesel,Manual,1,Hyderabad,3.00
1,cardekho_2025,CarDekho,2025,Ashok Leyland,Stile,LX OPTIONAL,lx o,2015,1000,Diesel,Manual,3,Other,9.00
2,cardekho_2026,CarDekho,2026,Aston Martin,DB11,V12,v12,2017,4700,Petrol,Automatic,1,Kolkata,239.00
3,cardekho_2026,CarDekho,2026,Aston Martin,Rapide,S V12,s v12,2017,7000,Petrol,Automatic,0,Nashik,250.00
4,cardekho_2026,CarDekho,2026,Audi,A3,35 TDI PREMIUM PLUS,35 plus premium tdi,2014,74000,Diesel,Automatic,0,Hyderabad,12.99


## 1. What's in the data

In [2]:
df.groupby(['Platform', 'Listing_Year']).size().rename('listings')

Platform  Listing_Year
CarDekho  2025             7512
          2026            12471
Cars24    2025             8008
          2026             6598
Spinny    2025             5731
Name: listings, dtype: int64

In [3]:
df['Selling_Price'].describe().round(2)

count    40320.00
mean         8.07
std         10.52
min          0.50
25%          3.55
50%          5.68
75%          9.19
max        450.00
Name: Selling_Price, dtype: float64

In [4]:
df.groupby('Brand')['Selling_Price'].agg(['count', 'median']).sort_values('count', ascending=False).head(12)

,count,median
Brand,,
Maruti Suzuki,9379,4.500
Hyundai,6997,5.010
Tata,3521,6.200
Honda,3335,4.950
Mahindra,2647,8.510
Renault,1889,3.800
Kia,1872,10.750
Toyota,1626,8.160
Ford,1501,4.200


## 2. How price falls with age (median, lakhs)

In [5]:
d = add_features(df)
d[d.Age <= 15].groupby('Age')['Selling_Price'].median().round(2).to_frame('median_price').T

Age,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
median_price,10.05,10.5,9.99,9.4,8.25,7.33,6.5,5.65,5.0,4.47,3.85,3.45,3.0,2.59,2.0,1.6


## 3. Train & test the final model
80/20 split, LightGBM on log(price) averaged over 3 seeds.

In [6]:
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(d, d[TARGET], test_size=0.2, random_state=42)
model = CarPriceModel().fit(X_tr, y_tr)
pred = model.predict(X_te)
ape = np.abs(pred - y_te) / y_te
print(f'Average error {ape.mean():.1%} | median error {ape.median():.1%} | within ±10%: {(ape<=0.1).mean():.0%} | within ±20%: {(ape<=0.2).mean():.0%}')

Average error 11.3% | median error 7.7% | within ±10%: 60% | within ±20%: 84%


## 4. Which inputs matter most

In [7]:
imp = pd.Series(np.mean([m.booster_.feature_importance('gain') for m in model.models_], axis=0), index=FEATURES)
(imp / imp.sum() * 100).sort_values(ascending=False).round(1).to_frame('importance_%')

,importance_%
Model_Full,32.1
Age,25.9
Brand,19.2
Trim,8.6
Transmission,4.3
Kms_Driven,2.9
Owner,1.8
City,1.7
Kms_Per_Year,1.5
Platform,1.0


## 5. Error by price band

In [8]:
bands = pd.cut(y_te, [0, 3, 5, 8, 15, 30, 500], labels=['<3L', '3-5L', '5-8L', '8-15L', '15-30L', '30L+'])
pd.DataFrame({'listings': ape.groupby(bands).size(), 'median_error_%': (ape.groupby(bands).median() * 100).round(1)})

,listings,median_error_%
Selling_Price,,
<3L,1529,11.5
3-5L,2028,7.9
5-8L,2037,6.3
8-15L,1767,6.7
15-30L,508,7.8
30L+,195,8.7


## 6. Full comparison
`python src/train.py` runs 5-fold cross-validation for every candidate model and saves the best one for the app:

In [9]:
pd.DataFrame({k: {**{'cv_mape': v['cv']['mape_pct']}, **{'test_'+m: v['test'][m] for m in ['mape_pct','r2','mae_lakhs']}} for k, v in json.load(open('models/metrics.json'))['models'].items()}).T

,cv_mape,test_mape_pct,test_r2,test_mae_lakhs
Ridge (linear baseline),14.21,13.51,0.7881,1.123
"LightGBM, no variant/city",14.62,13.92,0.8524,1.093
LightGBM (1 seed),12.19,11.42,0.8504,0.936
LightGBM (3-seed average),12.11,11.34,0.8488,0.930
